# Generate the result in RQ4

This script generates the results presented in the RQ4 section of the paper. The script reads the data from the `data/rq4-greybox` folder and generates the results presented in the paper.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
import multiprocessing as mp
import re
from scipy.stats import wilcoxon

# Directory where the coverage records are stored
directory = "data/rq4-greybox"

## Helper functions

In [3]:
# Record class
# exec_per_time: the average number of executions per time unit during the
#                fuzzing campaign
# p_emp_min: the minimum value of the empirical discovery probability
# data_done: the coverage records
@dataclass
class Record:
    exec_per_time: float
    p_emp_min: float
    data_done: pd.DataFrame


# Parse the record file and return a Record object
def get_record_obj(data_path):
    data = pd.read_csv(data_path, sep=", ", engine="python")
    data["done"] = data["done"].astype(bool)
    data["time"] = data["time"] / 1000 / 60
    # if #singletons or #sglt_clusts is 0, set it to 1
    data["#singletons"] = data["#singletons"].replace(0, 1)
    data["#sglt_clusts"] = data["#sglt_clusts"].replace(0, 1)
    data_done = data[data["done"]]
    # cut off the data after 24 hours
    data_done = data_done[data_done["time"] < 24 * 60]
    data_done.loc[:, "#execs"] = data_done["#execs"].astype(int)

    exec_per_time = data_done.iloc[-1]["#execs"] / data_done.iloc[-1]["time"]
    p_emps = data_done["#foundnew"] / data_done["#execs"]
    p_emps_min = np.min(p_emps[p_emps > 0])

    return Record(exec_per_time, p_emps_min, data_done)


# Given the time t_m, get the empirical discovery probability and
# the discovery probability estimated by the Good-Turing estimator (GT), Mean
# Local estimator with Good-Turing (MLG), and Mean Local estimator with our
# dependency-aware method (MLD)
def discovery(
    record_obj: Record, t_m: float, target: str, debug: bool = False
) -> float:
    exec_at_t = int(t_m * record_obj.exec_per_time)
    if exec_at_t == 0:
        return 0
    # find the row where #execs is the closest to exec_at_t
    for i, row in record_obj.data_done.iterrows():
        if row["#execs"] > exec_at_t:
            break
    row = record_obj.data_done.iloc[i - 1]
    assert row["done"]
    if debug:
        print(row)
    if target == "GT":
        ret = row["#singletons"] / exec_at_t
    elif target == "MLG":
        ret = row["ML_sglt"] / exec_at_t
    elif target == "MLD":
        ret = row["ML_sglt_clusts"] / exec_at_t
    elif target == "emp":
        ret = row["#foundnew"] / exec_at_t
        if ret == 0:
            ret = record_obj.p_emp_min
    else:
        raise ValueError(f"Invalid target: {target}")
    return ret


# Get the analyzable format of the estimation results
def get_plot_df(record_obj: Record, x_scale: str, id: str) -> pd.DataFrame:
    max_time = 24 * 60
    print(f"{id=}, {max_time=}", flush=True)
    if x_scale == "lin":
        x = np.linspace(0, max_time, 100)
    elif x_scale == "log":
        x = np.logspace(0, np.log10(max_time), 100)
    else:
        raise ValueError(f"Invalid x_scale: {x_scale}")
    # limit x to 24 hours
    x = x[x <= min(24 * 60, record_obj.data_done["time"].max())]
    print(f"{id=} processing start", flush=True, end="\r")
    y_emp = [discovery(record_obj, t, "emp") for t in x]
    print(f"{id=} emp done", flush=True, end="\r")
    y_GT = [discovery(record_obj, t, "GT") for t in x]
    print(f"{id=} GT done", flush=True, end="\r")
    y_MLG = [discovery(record_obj, t, "MLG") for t in x]
    print(f"{id=} MLG done", flush=True, end="\r")
    y_MLD = [discovery(record_obj, t, "MLD") for t in x]
    print(f"{id=} MLD done", flush=True, end="\r")
    plot_df = pd.DataFrame(
        {
            "time": x,
            "p_emp": y_emp,
            "p_GT": y_GT,
            "p_MLG": y_MLG,
            "p_MLD": y_MLD,
        }
    )
    plot_df = plot_df.melt(
        id_vars=["time"],
        value_vars=["p_emp", "p_GT", "p_MLG", "p_MLD"],
        value_name="Pr",
        var_name="Esti",
    )
    print(f"{id=} done", flush=True)
    plot_df["id"] = id
    return plot_df

In [4]:
# Read fuzzer_stats.csv files, which contains the summary of the
# fuzzing campaign, to get the repetition information
def read_fuzzer_stats(directory):
    """Reads the fuzzer_stats.csv file from the given directory."""
    file_path = os.path.join(directory, "fuzzer_stats.csv")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_csv(file_path)
    return df


def count_indices_for_subject(df, subject):
    """Returns the number of unique indices for a given subject."""
    if "subject" not in df.columns or "index" not in df.columns:
        raise ValueError(
            "DataFrame must contain 'subject' and 'index' columns."
        )

    return sorted(
        [int(i) for i in df[df["subject"] == subject]["index"].unique()]
    )

## Parse the record to generate the data for analysis

In [5]:
# get_record_obj(os.path.join(directory, "data/freetype2_0_records.csv"))
datas = [
    data
    for data in os.listdir(os.path.join(directory, "records"))
    if data.endswith("_records.csv")
]
subjects = np.unique([data.split("_")[0] for data in datas]).tolist()
print(f"subjects: {subjects}")

subjects: ['freetype2', 'jsoncpp', 'libjpeg', 'libpcap', 'libpng', 'libxml2', 'sqlite3', 'zlib']


In [6]:
df_fuzzer_stats = read_fuzzer_stats(directory)

for subject_name in subjects:
    index_count = count_indices_for_subject(df_fuzzer_stats, subject_name)
    print(
        f"Number of unique indices for subject '{subject_name}': {index_count}"
    )

Number of unique indices for subject 'freetype2': [0, 1, 2, 3, 4]
Number of unique indices for subject 'jsoncpp': [0, 1, 2, 3, 4]
Number of unique indices for subject 'libjpeg': [0, 1, 2, 3, 4]
Number of unique indices for subject 'libpcap': [0, 1, 2, 3, 4]
Number of unique indices for subject 'libpng': [0, 1, 2, 3, 4]
Number of unique indices for subject 'libxml2': [0, 1, 2, 3, 4]
Number of unique indices for subject 'sqlite3': [0, 1, 2, 3, 4]
Number of unique indices for subject 'zlib': [0, 1, 2, 3, 4]


In [7]:
# Generate data for each record and store it in the data directory
# the data is stored in the format of {subject}.csv
# If the data already exists, you can skip this step by setting is_generate
# to False.
is_generate = True
for subject in subjects:
    if is_generate or not os.path.exists(
        os.path.join(directory, f"records/{subject}.csv")
    ):
        print(f"Generating data for {subject}")
        idcs = count_indices_for_subject(df_fuzzer_stats, subject)
        print(
            [
                os.path.join(directory, f"records/{subject}_{i}_records.csv")
                for i in idcs
            ]
        )
        with mp.Pool(len(idcs)) as pool:
            record_objs = pool.map(
                get_record_obj,
                [
                    os.path.join(
                        directory, f"records/{subject}_{i}_records.csv"
                    )
                    for i in idcs
                ],
            )
        params = [
            (record_obj, "log", str(idx))
            for idx, record_obj in enumerate(record_objs, 1)
        ]
        with mp.Pool(len(params)) as pool:
            plot_dfs = pool.starmap(get_plot_df, params)
        plot_df = pd.concat(plot_dfs, ignore_index=True)
        plot_df["Esti"] = plot_df["Esti"].replace(
            {
                "p_emp": r"$\hat{m}_{emp}$",
                "p_GT": r"$\hat{m}_{GT}$",
                "p_MLG": r"$\hat{m}_{MLG}$",
                "p_MLD": r"$\hat{m}_{MLD}$",
            }
        )
        # save plot_df
        plot_df.to_csv(
            os.path.join(directory, "records/", f"{subject}.csv"), index=False
        )

Generating data for freetype2
['data/rq4-greybox/records/freetype2_0_records.csv', 'data/rq4-greybox/records/freetype2_1_records.csv', 'data/rq4-greybox/records/freetype2_2_records.csv', 'data/rq4-greybox/records/freetype2_3_records.csv', 'data/rq4-greybox/records/freetype2_4_records.csv']
id='2', max_time=1440id='3', max_time=1440id='1', max_time=1440id='4', max_time=1440id='5', max_time=1440




id='1' donedoneng startid='3' processing startid='5' processing startid='4' processing startid='1' emp done
id='5' donedone
id='3' donedone
id='4' donedone
id='2' donedone
Generating data for jsoncpp
['data/rq4-greybox/records/jsoncpp_0_records.csv', 'data/rq4-greybox/records/jsoncpp_1_records.csv', 'data/rq4-greybox/records/jsoncpp_2_records.csv', 'data/rq4-greybox/records/jsoncpp_3_records.csv', 'data/rq4-greybox/records/jsoncpp_4_records.csv']
id='1', max_time=1440id='2', max_time=1440id='3', max_time=1440id='5', max_time=1440id='4', max_time=1440




id='4' donedoneng startid='2' processi

## Discovery probability estimation accuracy (RQ4)

It generates
- Overall discovery probability estimation accuracy (Table 5 in the main paper)
- Discovery probability estimation accuracy at each time interval of the greybox fuzzing (Table 3 in the supplementary material)

In [8]:
# Aggregate the data
data = []
for subject in subjects:
    plot_df = pd.read_csv(os.path.join(directory, f"records/{subject}.csv"))
    sub_df = plot_df[["id", "time", "Esti", "Pr"]]
    for id in sub_df["id"].unique():
        sub_sub_df = sub_df[sub_df["id"] == id].copy()
        sub_sub_df["Esti"] = sub_sub_df["Esti"].replace(
            {
                r"$\hat{m}_{emp}$": "emp",
                r"$\hat{m}_{GT}$": "GT",
                r"$\hat{m}_{MLG}$": "MLG",
                r"$\hat{m}_{MLD}$": "MLD",
            }
        )
        sub_sub_df = (
            sub_sub_df.pivot(index="time", columns="Esti", values="Pr")
            .reset_index()
            .rename_axis(None, axis=1)
        )
        err_gt = np.abs(sub_sub_df["GT"] - sub_sub_df["emp"])
        err_mlg = np.abs(sub_sub_df["MLG"] - sub_sub_df["emp"])
        err_mld = np.abs(sub_sub_df["MLD"] - sub_sub_df["emp"])
        rel_err_gt = err_gt / sub_sub_df["emp"]
        rel_err_mlg = err_mlg / sub_sub_df["emp"]
        rel_err_mld = err_mld / sub_sub_df["emp"]

        time_intervals = [(1, 10), (10, 60), (60, 360), (360, 1440), (1, 1440)]
        for start_time, end_time in time_intervals:
            time_id = f"{start_time}-{end_time}"
            sub3_df = sub_sub_df[
                (sub_sub_df["time"] >= start_time)
                & (sub_sub_df["time"] <= end_time)
            ]
            time_diff = sub3_df["time"].diff().dropna()
            mid = lambda x: (x.iloc[1:] + x.iloc[:-1]) / 2
            weighted_avg = lambda x: np.sum(x.iloc[1:] * time_diff) / np.sum(
                time_diff
            )
            emp_avg = weighted_avg(mid(sub3_df["emp"]))
            err_gt_avg = weighted_avg(mid(err_gt))
            err_mlg_avg = weighted_avg(mid(err_mlg))
            err_mld_avg = weighted_avg(mid(err_mld))
            rel_err_gt_avg = weighted_avg(mid(rel_err_gt))
            rel_err_mlg_avg = weighted_avg(mid(rel_err_mlg))
            rel_err_mld_avg = weighted_avg(mid(rel_err_mld))
            data.append(
                {
                    "subject": subject,
                    "id": id,
                    "interval": time_id,
                    "emp_avg": emp_avg,
                    "err_gt_avg": err_gt_avg,
                    "err_mlg_avg": err_mlg_avg,
                    "err_mld_avg": err_mld_avg,
                    "rel_err_gt_avg": rel_err_gt_avg,
                    "rel_err_mlg_avg": rel_err_mlg_avg,
                    "rel_err_mld_avg": rel_err_mld_avg,
                }
            )
data_df = pd.DataFrame(data)
# assign very small value to 0 values
data_df.replace(0, 1e-20, inplace=True)
data_df

,subject,id,interval,emp_avg,err_gt_avg,err_mlg_avg,err_mld_avg,rel_err_gt_avg,rel_err_mlg_avg,rel_err_mld_avg
0,freetype2,1,1-10,0.035679,0.038029,10.714693,0.394739,0.998257,329.544650,11.699263
1,freetype2,1,10-60,0.032262,0.036234,3.669807,0.356697,0.999946,104.187670,10.445384
2,freetype2,1,60-360,0.028811,0.030591,1.212396,0.306582,0.999993,41.363326,11.492278
3,freetype2,1,360-1440,0.018507,0.018507,0.491203,0.253675,0.900684,27.383852,13.568925
4,freetype2,1,1-1440,0.022104,0.022104,0.845151,0.270964,0.929105,35.545924,12.965299
...,...,...,...,...,...,...,...,...,...,...
195,zlib,5,1-10,0.015908,0.022328,0.258888,0.211788,0.999728,59.234292,48.031462
196,zlib,5,10-60,0.004349,0.004351,0.266220,0.250916,0.996326,8443.959770,8354.315017
197,zlib,5,60-360,0.000023,0.000025,0.255144,0.254943,0.999086,10173.100497,10165.101985
198,zlib,5,360-1440,0.000023,0.000023,0.229829,0.229719,0.900498,9163.736965,9159.375604


### Overall discovery probability estimation accuracy (Table 5 in the main paper)

In [9]:
# average
data_df_avg = (
    data_df.replace([np.inf, -np.inf], np.nan)
    .dropna()
    .groupby(["subject", "interval"])
    .mean()
    .reset_index()
)
data_df_avg["ratio"] = data_df_avg["err_mld_avg"] / data_df_avg["err_mlg_avg"]
# subject order: "sqlite3", "freetype2", "libxml2", "libjpeg", "zlib",
# "libpcap", "jsoncpp", "libpng"
data_df_avg["subject"] = pd.Categorical(
    data_df_avg["subject"],
    categories=[
        "sqlite3",
        "freetype2",
        "libxml2",
        "libjpeg",
        "zlib",
        "libpcap",
        "jsoncpp",
        "libpng",
    ],
    ordered=True,
)
# interval order: ["1-10", "10-60", "60-360", "360-1440", "1-1440"]
data_df_avg["interval"] = pd.Categorical(
    data_df_avg["interval"],
    categories=["1-10", "10-60", "60-360", "360-1440", "1-1440"],
    ordered=True,
)
data_df_avg = data_df_avg.sort_values(["subject", "interval"])

In [10]:
total_df_avg = data_df_avg[data_df_avg["interval"] == "1-1440"]
total_df_avg = total_df_avg[
    [
        "subject",
        "emp_avg",
        "err_mlg_avg",
        "err_mld_avg",
        "rel_err_mlg_avg",
        "rel_err_mld_avg",
        "ratio",
    ]
]
# change the name of the columns
total_df_avg.columns = [
    "Subject",
    r"$\mu(\hat{m}_{\mathit{emp}})$",
    r"$\mu(\mathit{AE}_{MLG})$",
    r"$\mu(\mathit{AE}_{MLD})$",
    r"$\mu(\mathit{RE}_{MLG})$",
    r"$\mu(\mathit{RE}_{MLD})$",
    r"$\mu(\frac{\mathit{AE}_{MLD}}{\mathit{AE}_{MLG}})$",
]
total_df_avg = total_df_avg.reset_index(drop=True)
total_df_avg = total_df_avg.astype(str)
total_df_avg["Subject"] = total_df_avg["Subject"].map(lambda x: "\\" + x)
total_df_avg["Subject"] = total_df_avg["Subject"].map(
    lambda x: re.sub(r"\d+$", "", x)
)
total_df_avg.iloc[:, 1:4] = total_df_avg.iloc[:, 1:4].map(
    lambda x: f"{float(x):.2e}"
)

# Convert columns 3-5 to standard decimal format (2 decimal places)
total_df_avg.iloc[:, 4:7] = total_df_avg.iloc[:, 4:7].map(
    lambda x: f"{float(x):.2f}"
)
total_df_avg = total_df_avg.reset_index(drop=True)
display(total_df_avg)


,Subject,$\mu(\hat{m}_{\mathit{emp}})$,$\mu(\mathit{AE}_{MLG})$,$\mu(\mathit{AE}_{MLD})$,$\mu(\mathit{RE}_{MLG})$,$\mu(\mathit{RE}_{MLD})$,$\mu(\frac{\mathit{AE}_{MLD}}{\mathit{AE}_{MLG}})$
0,\sqlite,9.06e-02,3.30e+00,1.31e-01,33.80,1.40,0.04
1,\freetype,2.71e-02,8.83e-01,2.70e-01,32.91,11.35,0.31
2,\libxml,1.61e-02,1.80e+00,2.97e-01,94.52,18.10,0.16
3,\libjpeg,6.08e-03,2.85e-01,1.68e-01,205.40,156.21,0.59
4,\zlib,1.32e-03,2.20e-01,2.20e-01,8933.90,8929.59,1.00
5,\libpcap,2.53e-02,2.11e-01,1.85e-01,11.22,9.87,0.88
6,\jsoncpp,6.29e-04,2.34e-01,2.07e-01,8838.74,8552.86,0.88
7,\libpng,3.57e-03,2.42e-01,2.23e-01,2834.78,2764.06,0.92


### Discovery probability estimation accuracy at each time interval of the greybox fuzzing (Table 3 in the supplementary material)

In [11]:
data_df_sup = data_df_avg[~data_df_avg["interval"].isin(["1-1440"])]
data_df_sup = data_df_sup[
    [
        "subject",
        "interval",
        "emp_avg",
        "err_mlg_avg",
        "err_mld_avg",
        "rel_err_mlg_avg",
        "rel_err_mld_avg",
        "ratio",
    ]
]
data_df_sup["interval"] = pd.Categorical(
    data_df_sup["interval"],
    categories=["1-10", "10-60", "60-360", "360-1440"],
    ordered=True,
)
data_df_sup = data_df_sup.sort_values(["subject", "interval"])
# change the name of the columns
data_df_sup.columns = [
    "Subject",
    "Interval (min)",
    r"$\mu(\hat{m}_{\mathit{emp}})$",
    r"$\mu(\mathit{AE}_{MLG})$",
    r"$\mu(\mathit{AE}_{MLD})$",
    r"$\mu(\mathit{RE}_{MLG})$",
    r"$\mu(\mathit{RE}_{MLD})$",
    r"$\mu(\frac{\mathit{AE}_{MLD}}{\mathit{AE}_{MLG}})$",
]
data_df_sup = data_df_sup.reset_index(drop=True)
data_df_sup = data_df_sup.astype(str)
data_df_sup["Subject"] = data_df_sup["Subject"].map(lambda x: "\\" + x)
data_df_sup["Subject"] = data_df_sup["Subject"].map(
    lambda x: re.sub(r"\d+$", "", x)
)
data_df_sup.iloc[:, 2:5] = data_df_sup.iloc[:, 2:5].map(
    lambda x: f"{float(x):.2e}"
)

# Convert columns 3-5 to standard decimal format (2 decimal places)
data_df_sup.iloc[:, 5:8] = data_df_sup.iloc[:, 5:8].map(
    lambda x: f"{float(x):.2f}"
)
data_df_sup = data_df_sup.reset_index(drop=True)
display(data_df_sup)

,Subject,Interval (min),$\mu(\hat{m}_{\mathit{emp}})$,$\mu(\mathit{AE}_{MLG})$,$\mu(\mathit{AE}_{MLD})$,$\mu(\mathit{RE}_{MLG})$,$\mu(\mathit{RE}_{MLD})$,$\mu(\frac{\mathit{AE}_{MLD}}{\mathit{AE}_{MLG}})$
0,\sqlite,1-10,1.02e-01,3.71e+01,2.67e-01,394.76,2.74,0.01
1,\sqlite,10-60,1.05e-01,1.18e+01,1.53e-01,104.82,1.37,0.01
2,\sqlite,60-360,9.70e-02,4.93e+00,1.35e-01,47.73,1.33,0.03
3,\sqlite,360-1440,8.45e-02,2.06e+00,1.27e-01,22.75,1.41,0.06
4,\freetype,1-10,3.49e-02,1.31e+01,3.97e-01,434.67,13.03,0.03
5,\freetype,10-60,3.84e-02,4.09e+00,3.56e-01,112.71,9.94,0.09
6,\freetype,60-360,3.13e-02,1.24e+00,3.09e-01,40.39,10.53,0.25
7,\freetype,360-1440,2.41e-02,4.98e-01,2.52e-01,22.90,11.68,0.51
8,\libxml,1-10,3.43e-02,2.94e+01,4.59e-01,818.72,12.86,0.02
9,\libxml,10-60,2.27e-02,8.30e+00,4.21e-01,338.94,17.62,0.05
